# Assignment 2 Question 3

## d) Simulate the data

In [1]:
import numpy as np

np.random.seed(12345)

n, p = 200, 8  
X = np.random.randn(n, p)
true_beta =[2,-1.5,0,3,0,-2.5,1,5]

y = X @ true_beta + 0.5 * np.random.randn(n)  # Adding some noise

In [2]:
lambda_=0.2
tol=1e-6

The the maximum step size to ensure convergence is equal to $\frac{1}{\mu}=\frac{1}{\lambda\text{max}(\frac{1}{n}\mathbf{X'X}+\lambda)}$.

In [3]:
eigenvalues, eigenvectors = np.linalg.eig((X.T @ X)/n)
mu=max(eigenvalues)+lambda_
t=1/mu
print(f"The largest step size: {t:.4f}")

The largest step size: 0.6520


The upper bound on the suboptimality gap, $\mathcal{R}(\beta^{(k)})-\mathcal{R}(\beta^*)$, under the stopping condition, $||(\nabla_\beta\mathcal{R})(\beta)||_2\leq\epsilon$ is equal to $\frac{\epsilon^2}{2\tau}=\frac{(10^{-6})^2}{2(\lambda\text{min}(\frac{1}{n}\mathbf{X'X})+\lambda)}$.

In [4]:
tau=min(eigenvalues)+lambda_
max_gap=(tol**2)/(2*tau)
print(f"\u03c4: {tau:.4f}")
print("The upper bound on the suboptimality gap: ",max_gap) 

τ: 0.9223
The upper bound on the suboptimality gap:  5.421044018712684e-13


## f) Gradient decent

In [5]:
import pandas as pd
X_mean = np.mean(X, axis=0)
X_std = np.std(X, axis=0, ddof=0)
y_mean = np.mean(y)
    
X_std[X_std == 0] = 1  # To avoid division by zero for constant columns
X_scaled = (X - X_mean) / X_std
y_centered = y - y_mean

beta_star=np.linalg.inv(X_scaled.T@X_scaled+n*lambda_*np.identity(p)) @ (X_scaled.T@y_centered)

df=pd.DataFrame(true_beta,columns=["True Beta"])
df["Beta_star"]=beta_star

In [6]:
print(np.sqrt(beta_star.T@beta_star))



5.612989248904206


In [7]:
def gradient_decent(y,X,lambda_,beta_0,t,max_iter,tol):
    n=X.shape[0]
    beta_prev=beta_0
    gradient=-(X.T@y)/n
    for k in range(max_iter):
        beta_hat=beta_prev-t*gradient
        gradient=(X.T@X@beta_hat)/n-(X.T@y)/n+lambda_*beta_hat
        norm=np.linalg.norm(gradient)
        if (norm<=tol):
            return (beta_hat,k,norm)
        beta_prev=beta_hat

    return (beta_hat,k,norm)


In [8]:

coef,iterations,norm=gradient_decent(y_centered,X_scaled,lambda_,np.zeros(p),t,1000,tol)
df["Beta_k"]=coef

In [9]:
def R(beta,lambda_,y=y_centered,X=X_scaled):
    n=X.shape[0]
    return(((y-X@beta).T@(y-X@beta))/(2*n)+(lambda_/2)*(beta.T@beta))

gap=R(df["Beta_k"],lambda_)-R(df["Beta_star"],lambda_)

In [13]:
num=np.log((2*mu*(R(np.zeros(p),lambda_)-R(df["Beta_star"],lambda_)))/tol**2)
denom=-np.log(1-tau/mu)

suff_bound=num/denom

In [ ]:
import math

print("====== Gradient descent results ======")
print(" ")
print("Coefficients")
print(df)

print(" ")
print(f"Final gradient norm: {norm:.7f}")
print("Suboptimality gap: ", gap)
print("The upper bound on the suboptimality gap: ",max_gap)

print(" ")
print("Number of iterations (k): ",iterations+1)
print("Upper bound of suffiecient number of iterations: ",math.ceil(suff_bound))



====== Gradient descent results ======
 
Coefficients
   True Beta  Beta_star    Beta_k
0        2.0   1.715485  1.715485
1       -1.5  -1.073638 -1.073638
2        0.0   0.130419  0.130419
3        3.0   2.384244  2.384243
4        0.0  -0.123765 -0.123765
5       -2.5  -2.051740 -2.051740
6        1.0   0.670724  0.670723
7        5.0   4.127179  4.127179
 
Final gradient norm: 0.0000006
Suboptimality gap:  1.7674750552032492e-13
The upper bound on the suboptimality gap:  5.421044018712684e-13
 
Number of iterations (k):  17
Upper bound of suffiuecient number of iterations:  35


Gradient descent was able to converge to the exact minimum in 17 iterations. Notably, both the number of iterations (17) and the suboptimatilty gap, $\mathcal{R}(\beta^{(k)})-\mathcal{R}(\beta^*)=1.7674750552032492e-13$ were less than their theorical upper bounds of 35 iterations and  $\frac{\epsilon^2}{2\tau}=5.421044018712684e-13$, respectively, therefore confirming the theoretical results.  